# TRIBE v2: Colab demo with WhisperX workaround

Experimental notebook based on [Meta's TRIBE v2](https://github.com/facebookresearch/tribev2). The Python 3.11 workaround is **not yet validated by a full Colab run**. License of TRIBE v2: CC BY-NC 4.0. This is a research tool, not a predictor of virality.

1. Set **Runtime → Change runtime type → T4 GPU**.
2. Run installation cell, **restart runtime**, then run remaining cells in order.
3. Start with the **text example** at the bottom; video is optional.

Note: Llama 3.2 may require approved Hugging Face access.

## 1. Install

In [ ]:
!uv pip install "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"

## 2. Load model (after runtime restart)

In [ ]:
from tribev2.demo_utils import TribeModel, download_file
from tribev2.plotting import PlotBrain
from pathlib import Path
CACHE_FOLDER = Path("./cache")
CACHE_FOLDER.mkdir(exist_ok=True)
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder=CACHE_FOLDER)
plotter = PlotBrain(mesh="fsaverage5")
print("Model loaded")

## 3. WhisperX Python 3.11 workaround

This forces the WhisperX subprocess to use Python 3.11 and makes errors shorter. The fix still needs to be tested on Colab.

In [ ]:
import tribev2.eventstransforms as _evt
import torch
import pandas as pd

def _patched_transcript(wav_filename: Path, language: str) -> pd.DataFrame:
    import json, os, subprocess, tempfile
    codes = {"english":"en","french":"fr","spanish":"es","dutch":"nl","chinese":"zh"}
    if language not in codes:
        raise ValueError(f"Unsupported language: {language}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    with tempfile.TemporaryDirectory() as output_dir:
        cmd = ["uvx","--python","3.11","whisperx",str(wav_filename),
               "--model","large-v3","--language",codes[language],
               "--device",device,"--compute_type","float16",
               "--batch_size","16","--output_dir",output_dir,
               "--output_format","json"]
        if language == "english":
            cmd += ["--align_model","WAV2VEC2_ASR_LARGE_LV60K_960H"]
        env = {k:v for k,v in os.environ.items() if k != "MPLBACKEND"}
        result = subprocess.run(cmd, capture_output=True, text=True, env=env)
        if result.returncode:
            raise RuntimeError("WhisperX failed:\n" + "\n".join(result.stderr.splitlines()[-35:]))
        json_path = Path(output_dir) / f"{Path(wav_filename).stem}.json"
        transcript = json.loads(json_path.read_text())
    words=[]
    for i,segment in enumerate(transcript["segments"]):
        sentence=segment["text"].replace('"',"")
        for word in segment.get("words",[]):
            if "start" not in word or "end" not in word:
                continue
            words.append({"text":word["word"].replace('"',""),
                          "start":word["start"],"duration":word["end"]-word["start"],
                          "sequence_id":i,"sentence":sentence})
    return pd.DataFrame(words)

_evt.ExtractWordsFromAudio._get_transcript_from_audio = staticmethod(_patched_transcript)
print("WhisperX Python 3.11 workaround active")

## 4. Test text

Replace the English example with your own text after the first successful run. Text is converted to speech, so the output describes predicted response to **hearing**, not silently reading, a post. The original demo supports only certain languages.

In [ ]:
text = """To be or not to be, that is the question.
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune."""
text_path = CACHE_FOLDER / "sample.txt"
text_path.write_text(text)
df = model.get_events_dataframe(text_path=text_path)
display(df.head(8))

In [ ]:
preds, segments = model.predict(events=df)
print("Prediction shape:", preds.shape)

In [ ]:
fig = plotter.plot_timesteps(preds[:15], segments=segments[:15], cmap="fire", norm_percentile=99, vmin=.6, alpha_cmap=(0,.2), show_stimuli=True)

## 5. Optional video example

In [ ]:
video_path = CACHE_FOLDER / "sample_video.mp4"
download_file("https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4", video_path)
df_video = model.get_events_dataframe(video_path=video_path)
display(df_video.head(8))

In [ ]:
video_preds, video_segments = model.predict(events=df_video)
print("Video prediction shape:", video_preds.shape)